
<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />

## Worksheet 7.2 Deep Learning with RNNs

This notebook shows a common neural network architecture to detect malicious URLs using **RNNs**.

The task is to build a model that will be able to classify a URL as *malicious* or *benign*.

Libraries:
- [PyTorch](https://pytorch.org/) is used to build and train the LSTM model
- [string.printable](https://docs.python.org/3/library/string.html#string.printable) returns the printable symbols (digits, ascii_letters, punctuation, whitespace) used to encode each URL character
- pandas
- numpy

<div class="alert alert-info">
<strong>A note on hardware:</strong> PyTorch will use a GPU automatically when one is available, whether that is Apple Silicon (<code>mps</code>) or an NVIDIA card (<code>cuda</code>), and will otherwise fall back to the CPU. No additional plugins or installation steps are required. The cell below reports which device will be used.
</div>


In [ ]:
# Load Libraries - Make sure to run this cell!
import pandas as pd
import numpy as np
from string import printable
from sklearn import model_selection

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import warnings
warnings.filterwarnings("ignore")

## Selecting a Compute Device

PyTorch asks us to be explicit about where our data and model live. The cell below chooses the fastest device available and stores it in `device`. Later, both the model and every batch of data are moved onto that device with `.to(device)`.

In [ ]:
# PyTorch uses the GPU automatically if one is available.
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

## Load raw URL data
Extract the csv file from 
```
../data/url_data_mega_deep_learning.csv.zip
```
Then you can load the csv using the cell below. 

In [ ]:
## Load data URL

DATA_HOME = '../data/'
#df = pd.read_csv(DATA_HOME + 'url_data_mega_deep_learning.csv')
df = pd.read_csv(DATA_HOME + 'url_data_small_deep_learning.csv')

df.sample(n=25).head(5) 

## Pre-processing URL data
**Step 1**: Convert each raw URL string to integers. 

For a given string, if the characters that are contained in **printable**, they can be assigned a number (encoded) using the **printable.index()** 


In [ ]:
url_int_tokens = [[printable.index(x) + 1 for x in url if x in printable] for url in df.url]

# print out a few of these encoded characters
url_int_tokens[0][0:10]

## Step 2: Cut URL string at max_len or pad with zeros if shorter.

Because, we need for the inputs into a neural network to all be the same length.

Use the keras.preprocessing.pad_sequence method for this task

In [ ]:
max_len = 75

# Left-pad / left-truncate each sequence to max_len (matches keras pad_sequences defaults).
def pad_sequences(seqs, maxlen):
    out = np.zeros((len(seqs), maxlen), dtype=np.int64)
    for i, s in enumerate(seqs):
        s = s[-maxlen:]                 # truncate from the front if too long
        out[i, maxlen - len(s):] = s    # right-align, zeros on the left
    return out

features = pad_sequences(url_int_tokens, maxlen=max_len)

**Step 3:** Extract labels from the pandas dataframe and convert to a numpy array|

In [ ]:
target = np.array(df.isMalicious)

print('Dimensions of Features: ', features.shape,'\nDimensions of Targets: ', target.shape)

## Test/Train Split

In [ ]:
split_ratios = (0.7, 0.15, 0.15)  # Training, Validation, Test

features_train, features_temp, target_train, target_temp = model_selection.train_test_split(features, target, test_size=(1 - split_ratios[0]), stratify=target, random_state=42)
features_val, features_test, target_val, target_test = model_selection.train_test_split(features_temp, target_temp, stratify=target_temp, test_size=split_ratios[2] / (split_ratios[1] + split_ratios[2]), random_state=42)

# Labels stay as 1-D 0/1 values; the single-output model is trained with BCEWithLogitsLoss.
print('target_train shape:', target_train.shape)

In [ ]:
features_val.shape

## Architecture for an LSTM

In PyTorch a model is defined as a class that subclasses `nn.Module`, rather than as a stack of layers. Two methods do the work:

- `__init__` creates the layers the network will use.
- `forward` describes how an input tensor flows through those layers.

The network below mirrors the original design. An `Embedding` layer turns each character index into a dense vector, the `LSTM` reads the sequence and returns a final hidden state, `Dropout` provides regularisation, and a `Linear` layer produces a single output. The model returns a raw score (a logit) rather than a probability; we apply the sigmoid later, which is more numerically stable and is what the training loss expects.

In [ ]:
emb_dim = 32
max_vocab_len = 100          # printable chars are encoded as 1..100 (0 is padding)
lstm_output_size = 32

class URLClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Define the layers this network needs:
        #   - an Embedding layer. Index 0 is the padding value and the character
        #     tokens run from 1 to 100, so size it accordingly and set padding_idx=0.
        #   - an LSTM layer (use batch_first=True)
        #   - a Dropout layer
        #   - a Linear layer that produces a single output
        # Your code here...

    def forward(self, x):
        # Pass x through the embedding, then the LSTM. Take the final hidden state,
        # apply dropout, and pass it through the linear layer.
        # Return the raw logits with shape (batch,).
        # Your code here...
        pass

model = URLClassifier().to(device)
print(model)

## Train Model

Unlike Keras, PyTorch has no `fit` method; we write the training loop ourselves, which makes each step visible. For every batch of URLs we:

1. move the data to the chosen device,
2. clear the gradients left over from the previous step with `zero_grad`,
3. run a forward pass and compute the loss,
4. call `backward` to compute the gradients, and
5. update the weights with `optimizer.step`.

The loss function is `BCEWithLogitsLoss`, which combines the sigmoid and binary cross-entropy into a single, numerically stable step, so the model is trained on its raw logits. After each epoch we measure accuracy on the validation set to confirm the model is learning.

In [ ]:
nb_epoch = 3
batch_size = 512

def make_loader(features, target, shuffle):
    ds = TensorDataset(torch.tensor(features, dtype=torch.long),
                       torch.tensor(target, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(features_train, target_train, shuffle=True)
val_loader   = make_loader(features_val,   target_val,   shuffle=False)

criterion = nn.BCEWithLogitsLoss()          # expects raw logits + float 0/1 target
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def accuracy(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = (torch.sigmoid(model(xb)) > 0.5).float()
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

# Write the training loop. For each epoch, put the model in training mode and
# iterate over train_loader. For each batch (xb, yb):
#   1. move xb and yb to the device
#   2. zero the optimizer's gradients
#   3. run a forward pass and compute the loss with criterion
#   4. call backward on the loss
#   5. take an optimizer step
# After each epoch, print the validation accuracy using accuracy(val_loader).
# Your code here...

In [ ]:
# Evaluate the trained model on the held-out test set and print the accuracy.
test_loader = make_loader(features_test, target_test, shuffle=False)
# Your code here...

In [ ]:
torch.save(model.state_dict(), "lstm_URL_Classifier.pt")

## Making a Prediction

To classify a new URL we repeat exactly the same preprocessing used on the training data, then pass it through the trained model. We first switch the model to evaluation mode with `model.eval()`, which disables dropout, and wrap the call in `torch.no_grad()` so that PyTorch does not track gradients while we are only making a prediction. Applying the sigmoid to the model's output gives the probability that the URL is malicious.

In [ ]:
test_url_mal = "naureen.net/etisalat.ae/index2.php"
test_url_benign = "sixt.com/php/reservation?language=en_US"

url = test_url_mal

In [ ]:
# Step 1: Encode the raw URL string the same way as the training data
url_int_tokens = [[printable.index(x) + 1 for x in url if x in printable]]

# Step 2: Left-pad / truncate to max_len
processed_url = pad_sequences(url_int_tokens, maxlen=max_len)

In [ ]:
# Put the model in evaluation mode and, without tracking gradients, run
# processed_url through the model. Apply a sigmoid to turn the logit into the
# probability that the URL is malicious, and store it (as a numpy array) in target_proba.
# Your code here...

In [ ]:
def threshold_result(proba):
    if proba > 0.5:
        return "MALICIOUS!"
    else:
        return "benign"

In [ ]:
print("Test URL:\n", url, "\nis", threshold_result(target_proba[0]))